<a href="https://colab.research.google.com/github/Anushkapandey2110/Bandit/blob/main/SVM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import numpy as np
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

In [ ]:
#connecting to google drive
from google.colab import drive
drive.mount('/content/gdrive')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [ ]:
import os
#defining the path for test , train and validation datasets

base_dir = '/content/gdrive/MyDrive/Fish Eye Freshness.v2i.folder (1)'
train_dir = os.path.join(base_dir, 'train')
test_dir = os.path.join(base_dir, 'test')
val_dir = os.path.join(base_dir, 'valid')


In [ ]:
from tensorflow.keras.preprocessing import image_dataset_from_directory

In [ ]:
BATCH_SIZE = 32 #batch size kept less since less number of images

In [ ]:
train_dataset = image_dataset_from_directory(
    train_dir,
    shuffle=True,  #Shuffling the dataset since it is a good practice to ensure that the model does not learn any unintended patterns in the order of the training data.
    batch_size=BATCH_SIZE,
    image_size=(224, 224)  # MobileNetV2 expects 224x224 images

)

validation_dataset = image_dataset_from_directory(
    val_dir,
    shuffle=True,
    batch_size=BATCH_SIZE,
    image_size=(224, 224)  # MobileNetV2 expects 224x224 images

)

test_dataset = image_dataset_from_directory(
    test_dir,
    shuffle=False,
    batch_size=BATCH_SIZE,
    image_size=(224, 224)  # MobileNetV2 expects 224x224 images

)

Found 1080 files belonging to 2 classes.
Found 102 files belonging to 2 classes.
Found 51 files belonging to 2 classes.


In [ ]:
import tensorflow as tf

# Assuming train_dataset is already created
for image_batch, labels_batch in train_dataset.take(1):  # Take one batch from the dataset
    for img in image_batch:
        print(img.shape)
        break  # Print the shape of the first image in the batch


(224, 224, 3)


In [ ]:
dataset_name=test_dataset
class_names=test_dataset.class_names
print(class_names)

['fresh', 'non-fresh']


In [ ]:
feature_extractor = MobileNetV2(input_shape=(224, 224, 3),
                                include_top=False,
                                pooling='avg') #using the pre trained MobileNetV2 model trained on imagenet dataset,Excluding the top dense layers and leaving only the convolutional base of the network. This is done  to use the model as a feature extractor, and then add  SVM layer(s).
#By excluding the top layer, the model outputs the raw features instead of the final classification.
#pooling: Specifies the type of pooling operation to be applied to the output of the last convolutional layer.
#'avg': Applies global average pooling, which means it takes the average value of each feature map and outputs a single value per feature map. This reduces the dimensions of the output and results in a fixed-size vector regardless of the input size.
#Example: If the output of the last convolutional layer is a feature map of shape (7, 7, 1280), global average pooling will reduce this to a 1280-dimensional vector.

9406464/9406464 [==============================] - 0s 0us/step


In [ ]:
# Function to extract features and labels from a dataset
def extract_features_and_labels(dataset):
    features = []
    labels = []

    for images, label_batch in dataset:
        preprocessed_images = preprocess_input(images)
        features_batch = feature_extractor(preprocessed_images)
        features.append(features_batch.numpy())
        labels.append(label_batch.numpy())

    features = np.concatenate(features)
    labels = np.concatenate(labels)

    return features, labels

# Extract features and labels
train_features, train_labels = extract_features_and_labels(train_dataset)
validation_features, validation_labels = extract_features_and_labels(validation_dataset)
test_features, test_labels = extract_features_and_labels(test_dataset)

# Train an SVM model
svm_model = SVC(kernel='linear')
svm_model.fit(train_features, train_labels)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# Evaluate the model on the validation set
val_predictions = svm_model.predict(validation_features)
val_accuracy = accuracy_score(validation_labels, val_predictions)
val_precision = precision_score(validation_labels, val_predictions, average='weighted')
val_recall = recall_score(validation_labels, val_predictions, average='weighted')
val_f1 = f1_score(validation_labels, val_predictions, average='weighted')

print(f'Validation Accuracy: {val_accuracy:.4f}')
print(f'Validation Precision: {val_precision:.4f}')
print(f'Validation Recall: {val_recall:.4f}')
print(f'Validation F1 Score: {val_f1:.4f}')
print("\nValidation Classification Report:\n", classification_report(validation_labels, val_predictions, target_names=class_names))

# Evaluate the model on the test set
test_predictions = svm_model.predict(test_features)
test_accuracy = accuracy_score(test_labels, test_predictions)
test_precision = precision_score(test_labels, test_predictions, average='weighted')
test_recall = recall_score(test_labels, test_predictions, average='weighted')
test_f1 = f1_score(test_labels, test_predictions, average='weighted')

print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Precision: {test_precision:.4f}')
print(f'Test Recall: {test_recall:.4f}')
print(f'Test F1 Score: {test_f1:.4f}')
print("\nTest Classification Report:\n", classification_report(test_labels, test_predictions, target_names=class_names))

Validation Accuracy: 0.8529
Validation Precision: 0.8550
Validation Recall: 0.8529
Validation F1 Score: 0.8519

Validation Classification Report:
               precision    recall  f1-score   support

       fresh       0.88      0.78      0.83        46
   non-fresh       0.84      0.91      0.87        56

    accuracy                           0.85       102
   macro avg       0.86      0.85      0.85       102
weighted avg       0.85      0.85      0.85       102

Test Accuracy: 0.8431
Test Precision: 0.8431
Test Recall: 0.8431
Test F1 Score: 0.8431

Test Classification Report:
               precision    recall  f1-score   support

       fresh       0.85      0.85      0.85        27
   non-fresh       0.83      0.83      0.83        24

    accuracy                           0.84        51
   macro avg       0.84      0.84      0.84        51
weighted avg       0.84      0.84      0.84        51

